# Fine-tuning pour l’image

## 1. Préparation

**Cette partie est à exécuter en premier, de façon à télécharger les
bibliothèques, les données et les modèles nécessaires pour la suite du
TP**.

### Installation des bibliothèques (choisir l’une des options)

**Option 1** (solution propre mais un peu plus complexe, passez à
l’option 2 si vous ne comprenez rien)

1.  Si vous n’avez jamais utilisé `uv`, suivez les instructions
    d’installation ici
    <https://docs.astral.sh/uv/getting-started/installation/> (dans
    votre terminal, pas dans le notebook).
2.  Télécharger le fichier
    [`pyproject.toml`](https://schwander.isir.upmc.fr/enseignement/m2probafi_apprentissage/pyproject.toml)
    sur la page du cours.
3.  Mettre à jour l’environnement Python avec `uv sync` (dans votre
    terminal, pas dans le notebook).
4.  Fermer ce notebook et relancez Jupyter avec la commande
    `uv run jupyter notebook` (dans votre terminal, attention à bien
    être dans le répertoire avec le fichier `pyproject.toml`).

**Option 2** (à utiliser notamment avec Google Colab)

Copier-coller ce qui suit dans une cellule de ce notebook et exécuter
cette cellule:

    !pip install --index-url https://download.pytorch.org/whl/cpu "torch>=2.9.0"
    !pip install --index-url https://download.pytorch.org/whl/cpu "torchaudio>=2.9.0"
    !pip install --index-url https://download.pytorch.org/whl/cpu "torchvision>=0.24.0"
    !pip install "bitsandbytes>=0.48.2"
    !pip install "datasets>=4.4.1"
    !pip install "evaluate>=0.4.6"
    !pip install "gensim>=4.4.0"
    !pip install "jupyter>=1.1.1"
    !pip install "mlflow>=3.6.0"
    !pip install "nltk>=3.9.2"
    !pip install "peft>=0.18.0"
    !pip install "polars>=1.35.1"
    !pip install "rouge-score>=0.1.2"
    !pip install "transformers>=4.57.3"
    !pip install "trl>=0.25.1"
    !pip install "pytorch-lightning>=2.6.0"

### Téléchargement du dataset (exécuter ce qui suit)

In [ ]:
import torch
from torch import optim, nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import pytorch_lightning as pl

from datasets import load_dataset

import torchvision.transforms as T
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.models import vgg16, VGG16_Weights

In [ ]:
fashionmnist = load_dataset("zalando-datasets/fashion_mnist")
fashionmnist

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 60000
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 10000
    })
})

In [ ]:
fashionmnist["train"][92]["image"]

## 2. Définition d’un modèle simple

On définit un petit CNN tout simple. Au lieu d’utiliser directement
PyTorch, on utilise ici [Lightning](https://lightning.ai/docs/pytorch/)
qui nous facilitera la vie pour l’entraînement.

La façon de définir l’architecture ne change pas, mais on rajoute des
information pour l’entraînement directement dans la classe Python.

In [ ]:
class SmallCNN(pl.LightningModule):
    # PyTorch de base
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(p=0.5)
        self.fc1 = nn.Linear(3136, 128)
        self.fc2 = nn.Linear(128, 10)

    # PyTorch de base
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

    # Pour Lightning
    def configure_optimizers(self):
        self.criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.parameters(), lr=1e-3)
        return optimizer
    
    # Pour Lightning
    def training_step(self, batch, batch_idx):
        x = batch["image"]
        y = batch["label"]

        logits = self(x)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(1) == y).float().mean()
        self.log("train_loss", loss, prog_bar=True, on_epoch=True)
        self.log("train_acc", acc, prog_bar=True, on_epoch=True)
        return loss

    # Pour Lightning
    def validation_step(self, batch, batch_idx):
        x = batch["image"]
        y = batch["label"]

        logits = self(x)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True, on_epoch=True)
        self.log("val_acc", acc, prog_bar=True, on_epoch=True)

In [ ]:
print(SmallCNN())

SmallCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc1): Linear(in_features=3136, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)

On a chargé plus haut le dataset au format HuggingFace (avec la
bibliothèque `datasets`), il y a un peu de travail pour adapter ça au
format attendu par Lightning:

-   Avec `with_transform`, on modifie nos données pour passer d’une
    image à un tenseur. La transformation est faite de façon paresseuse
    (*lazy*), c’est à dire qu’on ne transforme pas tout le dataset
    durant un pré-traitement, mais uniquement quand on a besoin des
    données. À cet étape on pourrait rajouter de l’augmentation de
    données (voir par exemple la bibliothèque
    [Albumentations](https://albumentations.ai/docs/3-basic-usage/image-classification/%5D).
-   La function `hf_collate` nous permet de travailler sur des
    mini-batchs, on reçoit une liste Python d’exemples et on la
    transforme en un tenseur.
-   Les `DataLoader` est un élément de base de PyTorch, il permet
    d’itérer sur des batchs de données.

**Remarque** Il y a énormément de façons différentes de faire cette
partie.

In [ ]:
to_tensor = T.ToTensor()

def hf_collate(batch):
    imgs = [b["image"] for b in batch]
    labels = [b["label"] for b in batch]
    return {
        "image": torch.stack(imgs, dim=0).float(),
        "label": torch.tensor(labels, dtype=torch.long),
    }

class WrapperFashionMNIST(pl.LightningDataModule):
    def __init__(self, dataset):
        super().__init__()
        self.dataset = dataset

        to_tensor = T.ToTensor()

        self.train_ds = self.dataset["train"].with_transform(lambda ex: {"image": to_tensor(ex["image"]), "label": ex["label"]})
        self.val_ds = self.dataset["test"].with_transform(lambda ex: {"image": to_tensor(ex["image"]), "label": ex["label"]})

        self.train_ds.set_format(type="torch", columns=["image", "label"])
        self.val_ds.set_format(type="torch", columns=["image", "label"])

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=64, shuffle=True, collate_fn=hf_collate, num_workers=7)
    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=64, shuffle=False, collate_fn=hf_collate, num_workers=7)

In [ ]:
data = WrapperFashionMNIST(fashionmnist)

In [ ]:
for batch in data.val_dataloader():
    print(batch)
    break

{'image': tensor([[[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]],


        [[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]],


        [[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]],


        ...,


        [[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.

On peut maintenant lancer l’entraînement, c’est le module `Trainer` de
Lighthning qui se charge de la boucle:

In [ ]:
model = SmallCNN()
trainer = pl.Trainer(max_epochs=10, accelerator="auto", devices="auto")
trainer.fit(model, datamodule=data)

  | Name      | Type             | Params | Mode  | FLOPs
---------------------------------------------------------------
0 | conv1     | Conv2d           | 320    | train | 0    
1 | pool1     | MaxPool2d        | 0      | train | 0    
2 | conv2     | Conv2d           | 18.5 K | train | 0    
3 | pool2     | MaxPool2d        | 0      | train | 0    
4 | dropout   | Dropout          | 0      | train | 0    
5 | fc1       | Linear           | 401 K  | train | 0    
6 | fc2       | Linear           | 1.3 K  | train | 0    
7 | criterion | CrossEntropyLoss | 0      | train | 0    
---------------------------------------------------------------
421 K     Trainable params
0         Non-trainable params
421 K     Total params
1.687     Total estimated model params size (MB)
8         Modules in train mode
0         Modules in eval mode
0         Total Flops

## 3. Fine-tuning

On va commencer par charger des modèles pré-entraînés:

-   un VGG, un CNN relativement simple;
-   un ResNet, un CNN plus complexe, l’état de l’art jusqu’aux ViT.

In [ ]:
vgg = vgg16(weights=VGG16_Weights.DEFAULT)
resnet = resnet50(weights=ResNet50_Weights.DEFAULT)

Pour effectuer le fine-tuning, on a besoin de comprendre comment est
réalisée l’implémentation de l’architecture.

La première stratégie consiste à demander l’affichage l’architecture.
Que constate-on pour VGG ? Où et comment couper le réseau ?

In [ ]:
vgg

Une autre stragégie consiste à lire le code source de l’architecture:

-   VGG
    <https://github.com/pytorch/vision/blob/main/torchvision/models/vgg.py>
-   ResNet
    <https://github.com/pytorch/vision/blob/main/torchvision/models/resnet.py>

Une étape importante est de figer tous les paramètres du réseau que nous
ne voulons pas modifier lors de l’entraînement futur:

In [ ]:
for param in vgg.parameters():
    param.requires_grad = False

On peut maintenant remplacer la tête du réseau par un nouveau
classifier:

In [ ]:
model.classifier = nn.Linear(512, 2)

L’entraînement se fait ensuite de la façon habituelle.

Mêmes questions pour ResNet.